In [1]:
import pandas as pd 


In [2]:
df = pd.read_csv("IMDB Dataset.csv")

In [3]:
df.shape

(50000, 2)

In [4]:
df.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [5]:
df.isnull().sum()

review       0
sentiment    0
dtype: int64

In [6]:
df.drop_duplicates(inplace = True)

In [7]:
df.shape

(49582, 2)

#
# Pre - Processing :->

### 1. Converting to lowercase

In [8]:
df["review"] = df["review"].str.lower()

### 2. Removing the URLs

In [9]:
import re

In [10]:
def remove_urls(text):
    text = re.sub(r"http\S" , "" , text)
    return text

df["review"] = df["review"].apply(remove_urls)

### 3. Remove punctuations

In [11]:
def remove_punctuations(text):
    text = re.sub(r"[^A-Za-z0-9\s]" , "" , text)
    return text

df["review"] = df["review"].apply(remove_punctuations)

### 4. Remove HTML Tags

In [12]:
def remove_html(text):
    text = re.sub(r"<.*?>" , "" , text)
    return text

df["review"] = df["review"].apply(remove_html)

### 4. Remove the Stopwords

In [13]:
import nltk

In [14]:
nltk.download("punkt")
nltk.download("punkt_tab")
nltk.download("stopwords")

[nltk_data] Downloading package punkt to C:\Users\Pratush
[nltk_data]     Midya\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to C:\Users\Pratush
[nltk_data]     Midya\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to C:\Users\Pratush
[nltk_data]     Midya\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [15]:
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords

In [16]:
def remove_stopwords(text):
    tokens = word_tokenize(text)
    stop_words = stopwords.words("english")

    for word in tokens:
        if word in stop_words:
            text = text.replace(word , "" )

    return text

df["review"] = df["review"].apply(remove_stopwords)

In [17]:
df.head()

,review,sentiment
0,e revewers nted wtchg 1 oz epode ll ho...,positive
1,wderful ltle producti br br filming techniqu...,positive
2,thought ths wderful wy spend tme o hot s...,positive
3,bsclly res fmly lttle boy jke thks res zom...,negative
4,petter mtte love time mey vully stunng fi...,positive


### 6. Stemming 

In [20]:
from nltk.stem import PorterStemmer 

In [23]:
def stemming(text) :
    PS = PorterStemmer()
    stemmed_words = []

    tokens = word_tokenize(text)
    for token in tokens :
        stemmed_token = PS.stem(token)
        
        stemmed_words.append(stemmed_token)

    return " ".join(stemmed_words)

df["review"] = df["review"].apply(stemming)

In [24]:
df.head()

,review,sentiment
0,e revew nted wtchg 1 oz epod ll hook y rght ex...,positive
1,wder ltle producti br br film techniqu unssum ...,positive
2,thought th wder wy spend tme o hot summer week...,positive
3,bsclli re fmli lttle boy jke thk re zomb close...,negative
4,petter mtte love time mey vulli stunng film wt...,positive


### 7. Encoding 

In [25]:
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()

In [29]:
df["sentiment"] = le.fit_transform(df["sentiment"])

In [31]:
y = df["sentiment"]

In [32]:
y.head()

0    1
1    1
2    1
3    0
4    1
Name: sentiment, dtype: int64

### 8. Vectorization 

In [34]:
df.head()

,review,sentiment
0,e revew nted wtchg 1 oz epod ll hook y rght ex...,1
1,wder ltle producti br br film techniqu unssum ...,1
2,thought th wder wy spend tme o hot summer week...,1
3,bsclli re fmli lttle boy jke thk re zomb close...,0
4,petter mtte love time mey vulli stunng film wt...,1


In [35]:
from sklearn.feature_extraction.text import TfidfVectorizer

In [36]:
vectorizer = TfidfVectorizer(max_features=5000)

X = vectorizer.fit_transform(df["review"])

In [37]:
X

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 4062036 stored elements and shape (49582, 5000)>

In [49]:
X.shape

(49582, 5000)

#
# Dataset & Data Loaders :->

In [42]:
from sklearn.model_selection import train_test_split

In [44]:
X_train , X_test , y_train , y_test = train_test_split(
    X ,
    y , 
    test_size = 0.2 , 
    random_state = 42
)

In [46]:
X_train.shape

(39665, 5000)

In [47]:
X_test.shape

(9917, 5000)

In [54]:
import torch
from torch.utils.data import Dataset, DataLoader , TensorDataset

In [52]:
X_train = X_train.toarray()

In [53]:
X_test = X_test.toarray()

In [58]:
train_set = TensorDataset(
    torch.from_numpy(X_train).float(),
    torch.from_numpy(y_train.values).float()
)

In [59]:
test_set = TensorDataset(
    torch.from_numpy(X_test).float(),
    torch.from_numpy(y_test.values).float()
)

In [60]:
train_loader = DataLoader(
    train_set ,
    shuffle = True ,
    batch_size = 64
)

In [61]:
test_loader = DataLoader(
    test_set ,
    shuffle = True ,
    batch_size = 64
)

#
# Build RNN :->

In [75]:
import torch.nn as nn
import torch.optim as optim

In [73]:
class RNN(nn.Module) :
    def __init__(self , input_size , hidden_size = 128 , num_layer = 1) :
        super().__init__() 

        self.hidden_size = hidden_size 
        self.num_layer = num_layer

        self.rnn = nn.RNN( input_size , hidden_size , num_layer , batch_first = True )

        self.fc = nn.Linear(hidden_size , 1)

    def forward(self , X) :
        h0 = torch.zeros(self.num_layer , X.size(0) , self.hidden_size)
        out , _ = self.rnn(X , h0)

        out = self.fc(out[: , -1 , :])
        return out 

In [76]:
input_size = X_train.shape[1]

model = RNN(input_size)

criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters())

#
# Trainning The RNN :->

In [77]:
epochs = 10

for epoch in range(epochs):
    model.train()

    for Xb, yb in train_loader:
        optimizer.zero_grad()

        Xb = Xb.unsqueeze(1)
        
        outputs = model(Xb) 

        outputs = torch.sigmoid(outputs.squeeze()) 

        loss = criterion(outputs, yb)
        loss.backward()
        optimizer.step() 

    print(f"epoch = {epoch+1}/{epochs} and loss = {loss.item()}")

epoch = 1/10 and loss = 0.20570965111255646
epoch = 2/10 and loss = 0.2771606147289276
epoch = 3/10 and loss = 0.26379868388175964
epoch = 4/10 and loss = 0.19069188833236694
epoch = 5/10 and loss = 0.13690435886383057
epoch = 6/10 and loss = 0.2555603086948395
epoch = 7/10 and loss = 0.24052007496356964
epoch = 8/10 and loss = 0.296360582113266
epoch = 9/10 and loss = 0.19059911370277405
epoch = 10/10 and loss = 0.16838964819908142


In [79]:
model.eval()

with torch.no_grad():
    correct_vals = 0
    tot_vals = 0
    
    for Xb, yb in test_loader:
        Xb = Xb.unsqueeze(1)

        outputs = model(Xb)
        predicted = (torch.sigmoid(outputs.squeeze()) > 0.5).float()

        tot_vals += yb.size(0)
        correct_vals += (predicted == yb).sum().item()

    print(f"accuracy = {correct_vals/tot_vals*100}")

accuracy = 85.59040032267824
